# SHAP Local Explanation (External Validation)

## Purpose
Compute SHAP-based local explanation for representative binding poses
in the external validation case (PDB ID: 9UPD, ligand: nateglinide).
Produces Figure 8 in the manuscript.

## Method
SHAP contributions are computed from a final model (seed 42) trained on
all 31 internal complexes. Two poses are analyzed:
- **Pose 08** (false negative under DockF; corrected by DockF+MDF)
- **Pose 07** (false positive under DockF; corrected by DockF+MDF)

Note: In the paper, poses are labeled by CNN pose score order (GNINA default).
Pose 07 and Pose 08 refer to the 7th and 8th poses ranked by CNN pose score.

## Input
- CSV table with features for the external validation complex
- Final models:
  - `final_model/seed_42/lgb_ds/label_2p5_final/model.txt` (DockF)
  - `final_model/seed_42/lgb_ds_md/label_2p5_final/model.txt` (DockF+MDF)

## Output
- Figure 8: `predictions/seed_42/shap_summary.{pdf,png}`

## Run Order
1. Train final model (seed_42) using notebook 05 with final_model mode
2. Run external validation feature extraction
3. Run this notebook

In [ ]:
# ================================================================
# Cell 1: Libraries
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "regular",
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})


In [ ]:
# ================================================================
# Cell 2: Configuration
# ================================================================

# ── Configuration ──────────────────────────────────────────────────────────
# Update paths to match your local directory structure before running.
PROJECT_ROOT = Path("/path/to/predict")
FEATURE_CSV  = PROJECT_ROOT / "output" / "feature_table.csv"
MODEL_ROOT   = PROJECT_ROOT / "final_model"
OUT_ROOT     = PROJECT_ROOT / "output" / "predictions" / "seed_42"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED_TAG = "seed_42"      # final model seed tag
PDB_ID   = "9UPD"
TARGET   = "label_2p5"   # RMSD 2.5 Angstrom criterion (as in Figure 8)
EXPS     = [("ds", "DockF"), ("ds_md", "DockF+MDF")]

# Poses analyzed (paper notation: pose07, pose08 by CNN pose score order)
# Pose 08: false negative under DockF, corrected by DockF+MDF
# Pose 07: false positive under DockF, corrected by DockF+MDF
LIGANDS       = ["pose08", "pose07"]
POSE_INDEX_MAP = {"pose08": 8, "pose07": 7}  # for plot titles

DS_FEATURES = ["vina_affinity", "cnn_affinity", "cnn_pose_score"]
TOP_N_MD    = 10   # top features to show for DockF+MDF model
# ── End Configuration ──────────────────────────────────────────────────────


In [ ]:
# ================================================================
# Cell 3: Plot helpers
# ================================================================

PALETTE = {"DockF": "#ff7f0e", "MDF": "#1f77b4", "LigF": "#2ca02c"}
FIGSIZE = (14, 7.2)
HEIGHT_RATIOS = (0.35, 1.10)
BAR_H    = 0.55
FONT_TITLE = 15
FONT_LABEL = 13
FONT_TICK  = 11
WRAP_WIDTH = 28
Y_PAD      = 0.15

LABEL_MAP = {
    "cnn_affinity": "DockF: CNN affinity",
    "cnn_pose_score": "DockF: CNN pose score",
    "vina_affinity": "DockF: Vina affinity",
    "md_contact_per_atom_mean": "MDF: contacts/atom (mean)",
    "md_prolif_any_contact_count_mean": "MDF: any contact count (mean)",
    "md_mmgbsa_delta_total": r"MDF: MM/GBSA $\Delta G_{\mathrm{bind}}$ (mean)",
    "md_mmpbsa_delta_total": r"MDF: MM/PBSA $\Delta G_{\mathrm{bind}}$ (mean)",
    "md_distance_min": "MDF: ligand-binding-site distance (min)",
    "md_prolif_hbond_freq": "MDF: H-bond frequency",
    "md_prolif_ionic_count_mean": "MDF: ionic contact count (mean)",
    "md_rg_std": "MDF: radius of gyration (SD)",
    "md_contact_unique_residues": "MDF: unique contact residues",
    "md_eelec_slope": "MDF: electrostatic energy (slope)",
    "md_eelec_drift": "MDF: electrostatic energy (drift)",
}

def wrap_label(label, max_length=WRAP_WIDTH):
    if not label:
        return ""
    label = label.replace("DS: ", "DockF: ").replace("MD: ", "MDF: ")
    if len(label) <= max_length:
        return label
    if "$" in label and len(label.split("$")[0].strip()) <= 15:
        return label
    for i in range(max_length, max(0, max_length - 12), -1):
        if i < len(label) and label[i] in [" ", ":"]:
            return label[:i] + "\n" + label[i + 1:].lstrip()
    return label[:max_length] + "\n" + label[max_length:]

def format_label(name):
    return wrap_label(LABEL_MAP.get(name, name))

def feature_color(name):
    if name in DS_FEATURES:
        return PALETTE["DockF"]
    if str(name).startswith("md_"):
        return PALETTE["MDF"]
    return PALETTE["LigF"]

def load_model_and_features(experiment, target):
    base = MODEL_ROOT / SEED_TAG / f"lgb_{experiment}" / f"{target}_final"
    model_path = base / "model.txt"
    feat_path  = base / "features.csv"
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")
    booster = lgb.Booster(model_file=str(model_path))
    feat_order = pd.read_csv(feat_path).iloc[:, 0].astype(str).tolist()
    return booster, feat_order

def find_pose(df, pdb_id, ligand_id):
    mask = (df["pdb_id"].astype(str) == str(pdb_id)) & (df["ligand_id"].astype(str) == str(ligand_id))
    rows = df[mask]
    if len(rows) == 0:
        raise ValueError(f"Pose not found: pdb_id={pdb_id}, ligand_id={ligand_id}")
    return rows.iloc[0]

def compute_shap(booster, row, feat_order):
    vals = [float(row.get(f, 0.0)) if not pd.isna(row.get(f, 0.0)) else 0.0 for f in feat_order]
    X = np.array(vals, dtype=float).reshape(1, -1)
    contrib = np.asarray(booster.predict(X, pred_contrib=True)).reshape(-1)
    shap_vals = contrib[:-1]  # exclude bias term
    df = pd.DataFrame({"feature": feat_order, "shap_value": shap_vals})
    df["abs_shap"] = df["shap_value"].abs()
    return df.set_index("feature")

def style_axis(ax, n_bars, y_pad=Y_PAD):
    ax.axvline(0.0, color="0.55", ls="--", linewidth=1.0)
    ax.grid(axis="x", alpha=0.18)
    ax.tick_params(axis="x", labelsize=FONT_TICK)
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.set_ylim(n_bars - 0.5 + y_pad, -0.5 - y_pad)


In [ ]:
# ================================================================
# Cell 4: Compute SHAP values and build figure
# ================================================================

# Load feature data
df_features = pd.read_csv(FEATURE_CSV)

# Compute SHAP for all model/pose combinations
shap_tables = {}
for exp, _ in EXPS:
    booster, feat_order = load_model_and_features(exp, TARGET)
    for lig in LIGANDS:
        row = find_pose(df_features, PDB_ID, lig)
        shap_tables[(exp, lig)] = compute_shap(booster, row, feat_order)

# Compute unified x-axis limits across all panels
union_feats = set(DS_FEATURES)
for (exp, lig), df in shap_tables.items():
    if exp == "ds_md":
        top = df["abs_shap"].sort_values(ascending=False).head(TOP_N_MD).index.tolist()
        union_feats.update(top)

all_vals = [df.loc[f, "shap_value"] for df in shap_tables.values()
            for f in union_feats if f in df.index]
xmin, xmax = float(np.min(all_vals)), float(np.max(all_vals))
xpad = max(0.06 * (xmax - xmin), 0.03)
xlim = (xmin - xpad, xmax + xpad)

# Draw 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=FIGSIZE,
                          gridspec_kw={"height_ratios": list(HEIGHT_RATIOS)})
axes = axes.flatten()

order = [("ds", LIGANDS[0]), ("ds", LIGANDS[1]),
         ("ds_md", LIGANDS[0]), ("ds_md", LIGANDS[1])]
legend_handles = [
    Rectangle((0, 0), 1, 1, facecolor=PALETTE["DockF"], label="DockF"),
    Rectangle((0, 0), 1, 1, facecolor=PALETTE["MDF"], label="MDF"),
]

for ax, (exp, lig) in zip(axes, order):
    df = shap_tables[(exp, lig)]
    pose_idx = POSE_INDEX_MAP.get(lig, lig)
    model_label = "DockF+MDF" if exp == "ds_md" else "DockF"

    if exp == "ds":
        feats = sorted(DS_FEATURES, key=lambda f: df.loc[f, "abs_shap"] if f in df.index else 0, reverse=True)
    else:
        feats = df["abs_shap"].sort_values(ascending=False).head(TOP_N_MD).index.tolist()

    vals = [float(df.loc[f, "shap_value"]) if f in df.index else 0.0 for f in feats]
    n = len(feats)

    ax.barh(range(n), vals,
            color=[feature_color(f) for f in feats],
            height=BAR_H, edgecolor="black", linewidth=0.35)
    ax.set_yticks(range(n))
    ax.set_yticklabels([format_label(f) for f in feats], fontsize=FONT_TICK)
    ax.invert_yaxis()
    ax.set_xlim(xlim)
    ax.set_xlabel("SHAP value", fontsize=FONT_LABEL)
    rmsd_str = "RMSD 2.5 Angstrom" if TARGET == "label_2p5" else "RMSD 5.0 Angstrom"
    ax.set_title(f"Pose {pose_idx:02d} ({model_label} / {rmsd_str})",
                 fontsize=FONT_TITLE, pad=6)
    style_axis(ax, n_bars=n)

fig.legend(handles=legend_handles, loc="lower center",
           bbox_to_anchor=(0.5, 0.025), ncol=2, fontsize=13,
           frameon=True, borderpad=0.35, handlelength=2.2)
plt.tight_layout(pad=1.0, w_pad=1.4, h_pad=1.1, rect=[0, 0.045, 1, 1])


In [ ]:
# ================================================================
# Cell 5: Save output
# ================================================================

out_base = OUT_ROOT / "shap_summary"
fig.savefig(str(out_base) + ".pdf", bbox_inches="tight")
fig.savefig(str(out_base) + ".png", dpi=300, bbox_inches="tight")
print(f"[OK] Saved Figure 8: {out_base.name}.*")
plt.show()
